# 实验六｜HSV 颜色识别 + Week3 抓取 + 四桶垃圾分类

**流程**：摄像头单线程采集 → 畸变校正 → HSV 分割 → 开闭运算 → 轮廓 → 连续 6 帧稳定 → 按钮请求 → 像素转换到工作台 → Week3 风格抓取/抬升/转移/放置/复位。

**特别说明：** 四个桶的 X/Y/Z/Pitch 来自本次用户标定。提供的 Roll 为 `-945200.4° / 137329.9° / 137329.9° / 148789.0°`，不能视为可信的运动角度；**不会直接传给舵机或自动取模**。真实模式要求人工验证并填写 `roll_deg`。默认 `DRY_RUN=True`、不连接机械臂。此前若仍报 `KeyError:0`，必须先修复舵机配置。

**安全限制**：没有力觉、碰撞规划或物体高度估计；初次真机必须空载、低速、有人值守、可随时切断电源。XYZ 是否为 mm、是否在机械臂基坐标系，必须再次确认。

In [1]:
# 0. 依赖：在原课程 Jetson 环境运行，且当前目录可找到 arm5dof_uservo.py / camera.py / image2workspace.py
# 若缺依赖，在 Jupyter 的终端安装： python3 -m pip install opencv-python ipywidgets numpy
import os, sys, time, math, csv, threading, traceback
from pathlib import Path
from collections import deque
from dataclasses import dataclass
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display

SDK_DIR = next((p for p in [Path.cwd(), Path.cwd()/'4.机械臂OpenCV视觉处理实验', Path.cwd()/'2.机械臂实验']
                if (p/'arm5dof_uservo.py').exists() and (p/'camera.py').exists()), None)
print('课程 SDK 目录:', SDK_DIR or '未发现（可先运行演示模式）')
if SDK_DIR: sys.path.insert(0, str(SDK_DIR))
DRY_RUN = True   # 只模拟，不连接机械臂；必须显式修改此处才允许真机操作
MIN_AREA = 700
CONFIRM_FRAMES = 6
EMPTY_REARM_FRAMES = 8
SAFE_DZ = 55.0          # Z 安全抬升距离：必须实机确认
MOVE_T = 2.0
GRIPPER_T = 1.0
# 任选：None 检测整帧；或 (x1,y1,x2,y2) 限定工作台区域（像素）
ROI = None


课程 SDK 目录: /home/jetson/Notebook/4.机械臂OpenCV视觉处理实验


## 1. 目标桶位姿（坐标单位假设 mm；角度 deg）

红色数据中口述的「其他垃圾」与后面的「红色有害垃圾」不一致，按明确颜色—垃圾类型映射采用 **红色=有害垃圾**。Roll 留空并逐桶验证。**放置 Z 可能是悬停/示教点，而非适合张开夹爪的投放高度，必须先逐点验证。**

In [2]:
@dataclass(frozen=True)
class Pose:
    x: float; y: float; z: float; pitch_deg: float; roll_deg: float
    def xyz(self): return [self.x,self.y,self.z]

# 原始值仅用于人工核查：不可当作命令直接使用
RAW_ROLL = {'green': -945200.4, 'yellow': 137329.9, 'blue': 137329.9, 'red': 148789.0}
BINS = {
    'green':  {'label':'厨余垃圾', 'xyz':(122.8,114.8,122.8),  'pitch_deg':112.7, 'roll_deg':None},
    'yellow': {'label':'其他垃圾', 'xyz':(219.0,99.8,135.6),   'pitch_deg':106.5, 'roll_deg':None},
    'blue':   {'label':'可回收物', 'xyz':(207.8,-141.3,133.3),'pitch_deg':104.5, 'roll_deg':None},
    'red':    {'label':'有害垃圾', 'xyz':(127.4,-131.8,146.4),'pitch_deg':94.5,  'roll_deg':None},
}
# 仅当逐点示教/IK验证后再填写，例如： BINS['green']['roll_deg'] = 已验证的角度
# 重要：没有校准的 roll_deg 不允许真机抓取
for color, val in BINS.items(): print(color, val['label'], val['xyz'], 'pitch=',val['pitch_deg'], '原Roll=',RAW_ROLL[color])


green 厨余垃圾 (122.8, 114.8, 122.8) pitch= 112.7 原Roll= -945200.4
yellow 其他垃圾 (219.0, 99.8, 135.6) pitch= 106.5 原Roll= 137329.9
blue 可回收物 (207.8, -141.3, 133.3) pitch= 104.5 原Roll= 137329.9
red 有害垃圾 (127.4, -131.8, 146.4) pitch= 94.5 原Roll= 148789.0


## 2. 颜色算法：掩码、形态学、轮廓、中心点

检测到的中心 `(px,py)` 必须先经过去畸变图像对应的 `Image2Workspace` 变换；不能把像素坐标直接发给机械臂。

In [3]:
HSV_RANGES = {
    'red': [((0,80,50),(10,255,255)), ((170,80,50),(179,255,255))],
    'yellow': [((18,80,60),(38,255,255))],
    'green': [((40,60,35),(90,255,255))],
    'blue': [((90,75,45),(135,255,255))],
}
BGR = {'red':(0,0,255),'yellow':(0,255,255),'green':(0,255,0),'blue':(255,0,0)}
KERNEL = np.ones((5,5), np.uint8)

def detect_color(frame, selected, roi=None):
    hsv=cv2.cvtColor(frame,cv2.COLOR_BGR2HSV)
    mask=np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lower, upper in HSV_RANGES[selected]:
        mask |= cv2.inRange(hsv,np.array(lower,dtype=np.uint8),np.array(upper,dtype=np.uint8))
    if roi is not None:
        x1,y1,x2,y2=roi
        spatial=np.zeros_like(mask); spatial[max(0,y1):min(mask.shape[0],y2),max(0,x1):min(mask.shape[1],x2)]=255
        mask=cv2.bitwise_and(mask,spatial)
    mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,KERNEL)
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,KERNEL)
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    valid=[q for q in contours if cv2.contourArea(q)>=MIN_AREA]
    if not valid: return None, mask
    # 只选当前颜色最大区域，避免多个目标造成歧义
    contour=max(valid,key=cv2.contourArea)
    x,y,w,h=cv2.boundingRect(contour)
    return {'pixel':(x+w//2,y+h//2), 'box':(x,y,w,h), 'area':cv2.contourArea(contour)}, mask

def preview_detection(frame,color,det,stable):
    out=frame.copy()
    if det:
        x,y,w,h=det['box']; cx,cy=det['pixel']
        cv2.rectangle(out,(x,y),(x+w,y+h),BGR[color],2)
        cv2.circle(out,(cx,cy),5,(255,255,255),-1)
        cv2.putText(out,f'{color} area={det["area"]:.0f} stable={stable}/{CONFIRM_FRAMES}',
                    (max(5,x),max(25,y-8)),cv2.FONT_HERSHEY_SIMPLEX,.55,BGR[color],2)
    return out


## 3. Week3 运动序列：仅当明确开启真机并校验位姿后执行

单线程硬件拥有权：只允许下方的一个 worker 调用相机和机械臂；按钮仅提交请求，绝不直接发串口指令。为了避免运动期间再次读取画面，worker 在抓取期间暂停采集。首次推荐在 DRY_RUN 查看日志，不需要连接机械臂。

In [4]:
# 视觉标定抓取姿态初值：Week3 前几层的抓取姿态并不一定适合工作台散放目标。
# 下列数值只是 Week3 已使用的姿态参考；必须在真机首次执行前逐点确认。
PICK_PITCH_DEG = 114.5
PICK_ROLL_DEG = 144.5
# Image2Workspace 的 object_height 是工件高度；使用约定的工作台抓取参考高度。
# 可调整的微小接触偏移来自原实验 ArmApplication.GRAB_CUBE_DZ=5.0
PICK_Z_OFFSET_WS = 5.0

def checked_bin(color):
    cfg=BINS[color]
    roll=cfg['roll_deg']
    if roll is None or not np.isfinite(roll) or not (-180.0<=roll<=180.0):
        raise ValueError(f'{color} 桶的 roll_deg 未经过验证；真机动作已禁止')
    x,y,z=cfg['xyz']
    if not all(np.isfinite([x,y,z,cfg['pitch_deg'],roll])):
        raise ValueError('桶位姿包含非法数值')
    return Pose(x,y,z,cfg['pitch_deg'],roll)

def above(p):
    return Pose(p.x,p.y,p.z+SAFE_DZ,p.pitch_deg,p.roll_deg)

def move_to(arm,p,label):
    print(label, p)
    result=arm.set_tool_pose(p.xyz(), tool_pitch=math.radians(p.pitch_deg),
                             tool_roll=math.radians(p.roll_deg), T=MOVE_T,
                             is_soft_move=True,is_wait=True)
    if result is False: raise RuntimeError(label+' 逆运动学不可达，停止')

def run_pick_place(color, pixel, img2ws, app, arm, dry_run=True):
    if img2ws is None or app is None:
        if dry_run:
            print('[模拟] ', color, '目标像素', pixel, '目标桶', BINS[color]['label'], BINS[color]['xyz'])
            return
        raise RuntimeError('缺少相机九点标定 / 工作台到机械臂外参，禁止真机')
    wx,wy=img2ws.img2workspace(*pixel)
    wz=float(img2ws.object_height)+PICK_Z_OFFSET_WS
    ax,ay,az=app.tf_ws2arm(wx,wy,wz)
    pick=Pose(float(ax),float(ay),float(az),PICK_PITCH_DEG,PICK_ROLL_DEG)
    dest=checked_bin(color) if not dry_run else None
    print('像素:',pixel,'工作台:',(wx,wy,wz),'基坐标抓取:',pick,'投放桶:',BINS[color]['xyz'])
    if dry_run: return
    if arm is None: raise RuntimeError('未初始化机械臂')
    if pick.z<10 or above(pick).z>350:
        raise ValueError('抓取高度触发保守保护；请复核标定')
    # 与 Week3 保持一致：先上方 Approach，再竖直下降、抬升、Home 转运。
    arm.gripper_open(T=GRIPPER_T,is_wait=True)
    move_to(arm,above(pick),'Approach')
    move_to(arm,pick,'Pick')
    arm.gripper_close(T=GRIPPER_T,is_wait=True)
    move_to(arm,above(pick),'Lift')
    arm.home()
    move_to(arm,above(dest),'Transfer')
    move_to(arm,dest,'Place')
    arm.gripper_open(T=GRIPPER_T,is_wait=True)
    move_to(arm,above(dest),'Retreat')
    arm.home()


## 4. 交互界面：颜色选择、实时画面、一次性分拣、状态互锁

- **启动预览**：启动唯一的后台工作线程；检测稳定后状态 `READY`。
- **分拣一次**：记录一次操作请求，只有连续帧稳定才执行。按钮不会造成重复抓取。
- **停止**：停止后续新任务；**运动中不能立刻硬中断**，必须使用实体急停/断电处理紧急情况。
- **真机使用**：先停止线程，把 `DRY_RUN=False`，逐桶填写经过示教验证的 Roll，确认标定和 SDK 正常，再重新启动。禁止在运行中改成真机模式。

In [8]:
color_choice=widgets.Dropdown(options=[('绿色｜厨余垃圾','green'),('黄色｜其他垃圾','yellow'),('蓝色｜可回收物','blue'),('红色｜有害垃圾','red')],description='选择颜色')
btn_start=widgets.Button(description='启动摄像头',button_style='success')
btn_sort=widgets.Button(description='分拣一次',button_style='warning')
btn_stop=widgets.Button(description='停止预览',button_style='danger')
mode_label=widgets.HTML(value='<b>默认：DRY_RUN 模拟模式</b>')
state_label=widgets.HTML(value='状态：STOPPED')
image_widget=widgets.Image(format='jpeg',width=640)
log_box=widgets.Output(layout={'border':'1px solid #ddd','height':'180px','overflow':'auto'})
display(widgets.VBox([widgets.HBox([color_choice,btn_start,btn_sort,btn_stop]),mode_label,state_label,image_widget,log_box]))

runtime={'thread':None,'stop':threading.Event(),'state':'STOPPED','pending':False,'running_mode':None,
         'last_result':None,'error':None,'history':[]}
lock=threading.Lock()

def set_state(state, detail=''):
    with lock: runtime['state']=state
    state_label.value=f'<b>状态：{state}</b> {detail}'

def worker(mode):
    cam=None; arm=None; img2ws=None; app=None
    selected_prev=None; stable=0; empty_count=0; rearmed=True
    try:
        # 模拟模式优先使用课程 Camera；若不存在 SDK，退回 OpenCV 摄像头。
        if SDK_DIR:
            from camera import Camera
            from image2workspace import Image2Workspace
            cam=Camera(); cam.load_cam_calib_data()
            try: img2ws=Image2Workspace(cam)
            except Exception as ex:
                print('九点标定加载失败：',ex)
                if not mode: raise
        else:
            if not mode: raise RuntimeError('真机模式必须在课程 SDK 环境运行')
            cam=cv2.VideoCapture(0)
            if not cam.isOpened(): raise RuntimeError('摄像头打开失败')
        # 真机只在本线程初始化一次舵机，避免多进程抢串口。
        if not mode:
            from arm5dof_uservo import Arm5DoFUServo
            from arm_application import ArmApplication
            for c in BINS: checked_bin(c)
            arm=Arm5DoFUServo()  # 若遇 KeyError:0，检查舵机映射，禁止继续
            app=ArmApplication(arm, config_folder=str(SDK_DIR/'config'))
        elif img2ws is not None:
            # 纯数学坐标换算，不会控制机械臂
            from arm_application import ArmApplication
            class Stub: pass
            app=ArmApplication(Stub(), config_folder=str(SDK_DIR/'config'))
        set_state('READY','模拟' if mode else '真机')
        while not runtime['stop'].is_set():
            if SDK_DIR:
                ret,frame=cam.read()
                if ret: frame=cam.remove_distortion(frame)
            else: ret,frame=cam.read()
            if not ret: raise RuntimeError('图像读取失败')
            with lock:
                pending=runtime['pending']; selected=color_choice.value
            if selected!=selected_prev:
                stable=0; empty_count=0; rearmed=True; selected_prev=selected
                with lock: runtime['pending']=False
                pending=False
            det,mask=detect_color(frame,selected,ROI)
            stable=min(CONFIRM_FRAMES,stable+1) if det else 0
            empty_count=0 if det else empty_count+1
            if empty_count>=EMPTY_REARM_FRAMES: rearmed=True
            out=preview_detection(frame,selected,det,stable)
            ok,jpeg=cv2.imencode('.jpg',out,[int(cv2.IMWRITE_JPEG_QUALITY),75])
            if ok: image_widget.value=jpeg.tobytes()
            if pending and stable==CONFIRM_FRAMES and rearmed and det:
                with lock: runtime['pending']=False
                rearmed=False
                set_state('BUSY','机械臂动作期间暂停摄像头读取')
                success=False; error=''
                try:
                    run_pick_place(selected,det['pixel'],img2ws,app,arm,dry_run=mode)
                    success=True
                except Exception as ex:
                    error=str(ex); print('分拣失败:',traceback.format_exc())
                runtime['history'].append({'time':time.strftime('%Y-%m-%d %H:%M:%S'),
                    'color':selected,'bin':BINS[selected]['label'],'pixel':str(det['pixel']),
                    'mode':'dry_run' if mode else 'hardware','success':success,'error':error})
                stable=0
                if not success and not mode:
                    set_state('ERROR','动作失败，已禁止新任务；检查机械臂')
                    break
                set_state('READY','请移走旧物体后再次点击分拣')
            elif pending:
                set_state('CONFIRM',f'目标 {selected}：{stable}/{CONFIRM_FRAMES} 帧；等待重新布置目标' if not rearmed else f'{stable}/{CONFIRM_FRAMES} 帧')
            elif runtime['state'] not in ('READY','BUSY'):
                set_state('READY','等待分拣指令')
            time.sleep(.04)
    except Exception as ex:
        runtime['error']=traceback.format_exc(); set_state('ERROR',str(ex))
        with log_box: print(runtime['error'])
    finally:
        if cam is not None:
            try:
                if SDK_DIR: cam.capture.release()
                else: cam.release()
            except Exception: pass
        # 异常时不自动移动回零，以免未知碰撞状态下造成新的运动。
        if runtime['state']!='ERROR': set_state('STOPPED')

def on_start(_):
    t=runtime['thread']
    if t is not None and t.is_alive(): return
    runtime['stop']=threading.Event(); runtime['pending']=False
    runtime['running_mode']=bool(DRY_RUN)
    mode_label.value='<b>模式：'+('模拟（不移动机械臂）' if DRY_RUN else '真机（已启用机械臂）')+'</b>'
    runtime['thread']=threading.Thread(target=worker,args=(bool(DRY_RUN),),daemon=True)
    runtime['thread'].start()

def on_sort(_):
    with lock:
        if runtime['state'] in ('BUSY','ERROR','STOPPED'): return
        runtime['pending']=True
    set_state('DETECT','已提交一次分拣请求')

def on_stop(_):
    runtime['stop'].set()
    with lock: runtime['pending']=False
    if runtime['state']!='BUSY':set_state('STOPPING','等待摄像头线程关闭')
    else:state_label.value='状态：BUSY；停止请求将在动作结束后生效，急停请使用硬件开关'

btn_start.on_click(on_start)
btn_sort.on_click(on_sort)
btn_stop.on_click(on_stop)


## 5. 保存 8 次随机测试记录

运行 8 次测试后调用下方单元，将每次结果写入 CSV。模拟成功只是算法流程成功，不能等同于实际投放成功；真实成功与否还需要人工观察确认。

In [9]:
def export_records(path='week6_sort_records.csv'):
    cols=['time','color','bin','pixel','mode','success','error']
    with open(path,'w',newline='',encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=cols); writer.writeheader(); writer.writerows(runtime['history'])
    print('记录已保存:',Path(path).resolve(),'共',len(runtime['history']),'次')

# 需要时执行：export_records()


## 真机操作前的核对清单

1. 修复此前 `KeyError: 0`，并确认 `Arm5DoFUServo()` 初始化可靠；禁止同时运行旧 `grab_color_block.py` 或其他占用串口/摄像头的进程。
2. 重新读取四个桶的 **Roll**，确认度数正常，并对每个桶的 XYZ/Pitch/Roll 逐点检查 IK 可达、投放低位和上方安全点；只确认数值合理远远不够。
3. 九点标定 `vconfig`、`T_arm2ws.txt`、`object.yaml` 必须与摄像头和机械臂当前物理位置一致。
4. 当前抓取姿态使用 Week3 的初值 `114.5°/144.5°`，需要重新验证其对桌面物体的适配性；`SAFE_DZ` 与 `PICK_Z_OFFSET_WS` 也需要实测。
5. 先模拟，再逐一空载验证 Approach/Pick/Lift/Transfer/Place/Retreat；在工作区无人手伸入且有实体急停的情况下才允许带块抓取。
6. Notebook 中 `Stop` 不能中止正在阻塞的硬件命令；急停请使用设备实际急停开关。

**算法局限**：同色多目标只抓最大轮廓；没有追踪 ID。重复抓取由“一次性请求 + 缺失帧重置”约束；如镜头被机械臂遮挡、视角变化或移动后物体仍停在同一位置，必须人工核对。